In [3]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
import yaml 
from pathlib import Path

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

torch.set_float32_matmul_precision('medium')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [25]:
config_path = Path("config/binaural_attn/word_task_v10_main_feature_gain_config.yaml")
ckpt_path = Path("attn_cue_models/word_task_v10_main_feature_gain_config/checkpoints/epoch=1-step=24679-v1.ckpt")


config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 2
config['hparas']['batch_size'] = 32
config['corpus']['root'] = '/orcd/data/jhm/001/datasets/dataset_binaural_attn/v10/'

In [57]:
from typing import Optional, Dict

class StageFeatureExtractor(nn.Module):
    """
    Wraps BinauralAttentionCNN and extracts features at a chosen stage.
    """
    def __init__(self, backbone: nn.Module, target_stage: str):
        super().__init__()
        self.backbone = backbone
        self.target_stage = target_stage
        self._features = None

        # register forward hook at the chosen stage
        for name, module in self.backbone.named_modules():
            if name == target_stage:
                module.register_forward_hook(self._hook)
                break
        else:
            raise ValueError(f"Stage {target_stage} not found in backbone")

    def _hook(self, module, input, output):
        self._features = output

    def forward(self, x: torch.Tensor):
        _ = self.backbone(x)   # run whole forward pass
        return self._features  # feature map at target stage


class MultiClassifierModule(pl.LightningModule):
    """
    LightningModule that fits three classifiers on a frozen CNN stage.
    """
    def __init__(self, backbone, target_stage: str,
                 task_dict: Dict[str, int] = {"num_azim_classes": 512,
                                            "num_word_classes": 800,
                                            "num_semitone_classes": 200},
                 lr: float = 1e-3):
        super().__init__()
        self.save_hyperparameters(ignore=["backbone"])
        self.task_dict = task_dict
        self.task_names = list(task_dict.keys())  # Store ordered task names

        # feature extractor from stage
        self.feature_extractor = StageFeatureExtractor(backbone, target_stage)
        # freeze CNN params
        for p in backbone.parameters():
            p.requires_grad = False
        for p in self.feature_extractor.parameters():
            p.requires_grad = False

        # infer feature dimension
        with torch.no_grad():
            # size of binaural audio at 2.5 seconds (44.1kHz)
            dummy = torch.randn(1, 2, 110_250)   
            feats = self.feature_extractor(dummy)
            feat_dim = feats.view(1, -1).shape[1]

        # define independent classifiers with consistent naming
        for task, num_classes in task_dict.items():
            head = nn.Linear(feat_dim, num_classes)
            setattr(self, f"head_{task}", head)

    def forward(self, x):
        feats = self.feature_extractor(x)
        return feats.view(feats.size(0), -1)

    def training_step(self, batch, batch_idx, optimizer_idx):
        # Expect batch format: x, {task_name: target_tensor, ...}
        # e.g., x, {"num_azim_classes": azim_targets, "num_word_classes": word_targets, ...}
        x, targets = batch
        feats = self(x)
        
        # Get the task name corresponding to this optimizer
        task_name = self.task_names[optimizer_idx]
        
        # Get the appropriate head and target
        head = getattr(self, f"head_{task_name}")
        target = targets[task_name]
        
        # Compute loss
        logits = head(feats)
        loss = F.cross_entropy(logits, target)
        
        # Log with clean task name
        self.log(f"train_loss_{task_name}", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, targets = batch
        feats = self(x)
        
        total_loss = 0
        for task_name in self.task_names:
            head = getattr(self, f"head_{task_name}")
            target = targets[task_name]
            
            logits = head(feats)
            loss = F.cross_entropy(logits, target)
            
            self.log(f"val_loss_{task_name}", loss, prog_bar=True)
            total_loss += loss
            
        self.log("val_loss_total", total_loss, prog_bar=True)
        return total_loss

    def configure_optimizers(self):
        # Create one optimizer per task head
        optimizers = []
        for task_name in self.task_names:
            head = getattr(self, f"head_{task_name}")
            optimizer = torch.optim.Adam(head.parameters(), lr=self.hparams.lr)
            optimizers.append(optimizer)
        
        return optimizers

    def predict_step(self, batch, batch_idx):
        x = batch if isinstance(batch, torch.Tensor) else batch[0]
        feats = self(x)
        
        predictions = {}
        for task_name in self.task_names:
            head = getattr(self, f"head_{task_name}")
            logits = head(feats)
            predictions[task_name] = torch.softmax(logits, dim=1)
            
        return predictions

def list_stage_names(model: nn.Module):
    """
    Print all available module names inside a model so you can pick a target stage.
    """
    print("Available stages in model:\n")
    for name, module in model.named_modules():
        print(f"{name:30s} ({module.__class__.__name__})")

def list_act_names(model: nn.Module):
    """
    Print all available module names inside a model so you can pick a target stage.
    """
    print("Available stages in model:\n")
    name_list = []
    for name, module in model.named_modules():
        if 'ReLU' in module.__class__.__name__:
            print(f"{name:30s} ({module.__class__.__name__})")
            name_list.append(name)
    return name_list


def format_stage_name(name: str):
    """
    Convert a stage name to a valid Python attribute name.
    """
    if 'conv' in name:
        name = f"_orig_mod.model_dict.{name}.2" # 2 is the ReLU layer after conv
    elif 'fc' in name:
        name = "_orig_mod.relufc"
    return name

class ModelWithFrontEnd(nn.Module):
    def __init__(self,front_end, model):
        super().__init__()
        self.front_end = front_end
        self.model = model

    def forward(self, cue: torch.Tensor = None, 
                mixture: torch.Tensor = None,
                cue_mask_ixs: torch.tensor = None): 
        cue, mixture = self.front_end(cue, mixture)
        return self.model(cue, mixture, cue_mask_ixs)


In [59]:


from src.spatial_attn_lightning import BinauralAttentionModule  # import your CNN

module = BinauralAttentionModule.load_from_checkpoint(config=config, checkpoint_path=ckpt_path)
cochgram = module.coch_gram
cnn = module.model
backbone = ModelWithFrontEnd(cochgram, cnn)

train_dataloader = module.train_dataloader()
val_dataloader = module.val_dataloader()

act_names = list_act_names(backbone)
act_name_dict = {i+1: name for i, name in enumerate(act_names)}
act_name_dict[0] = 'cochlegram'

LAYER_IX = 5
layer_to_get = act_name_dict[LAYER_IX]  # change index to select different stage

if layer_to_get == 'cochlegram':
    feature_extractor = cochgram
else:
    feature_extractor = StageFeatureExtractor(backbone, layer_to_get)
feature_extractor.backbone.model.classification = nn.Identity()  # remove final classification layers



model = MultiClassifierModule(
    backbone=feature_extractor,
    target_stage=layer_to_get,
    task_dict={
        "num_azim_classes": 512,
        "num_word_classes": 800, 
        # "num_semitone_classes": 200
    },
    lr=1e-3
)

Using explicit dim specification for demeaning in audio transforms
Batch in dataloader = False
Using BinauralAuditoryAttentionCNN
v08 True
num_classes={'num_words': 800}
Model performing word task
Using singe gain function per layer
Conv block order: LN -> Conv -> ReLU
fc_attn: True
coch_affine: True
Using dataset BinauralAttentionDataset
OptimizedModule(
  (_orig_mod): BinauralAuditoryAttentionCNN(
    (model_dict): ModuleDict(
      (norm_coch_rep): LayerNorm((2, 40, 20000), eps=1e-05, elementwise_affine=True)
      (attn0): SimpleAttentionalGain()
      (conv_block_0): Sequential(
        (0): LayerNorm((2, 40, 20000), eps=1e-05, elementwise_affine=True)
        (1): Conv2d(2, 32, kernel_size=(2, 34), stride=(1, 1), bias=False)
        (2): ReLU()
      )
      (hann_pool_0): HannPooling2d()
      (attn1): SimpleAttentionalGain()
      (conv_block_1): Sequential(
        (0): LayerNorm((32, 20, 4992), eps=1e-05, elementwise_affine=True)
        (1): Conv2d(32, 64, kernel_size=(2, 14

ValueError: Stage model._orig_mod.model_dict.conv_block_4.2 not found in backbone